[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/04-recall-tuning/03-numeric_matching_ranges_and_thresholds.ipynb)

# Numeric Matching: Ranges and Thresholds

Everything in the previous two notebooks compared strings, character by character, fragment by fragment. Numbers need a different kind of comparison entirely. Asking "how similar are the characters in 49.99 and 52.00" is not a meaningful question. Asking "how close are these two prices" or "is this product newer than that year" is.

M|BOX has a dedicated set of numeric comparison modes for exactly this: proximity, strict equality, greater-than, and less-than. In this notebook you will:

1. Understand why numeric fields need their own comparison modes, and how those modes relate to the string modes you already know
2. Use `NUM_APPROX` to find values close to a target, with score decreasing as distance increases
3. Use `NUM_EXACT` for strict numeric equality
4. Use `NUM_GREATER` and `NUM_LOWER` to express one-sided thresholds, "at least this much" or "no more than this much"
5. Combine a numeric field with a text field in a single query

In [1]:
# !pip install mbox

## 1. Why numeric fields need their own modes

Take a look at how `TableRecallMode` is actually defined:

```python
class TableRecallMode(str, Enum):
    EXACT = "Exact"
    COMPLETE = "Complete"
    APPROX = "Approx"
    DETECT = "Detect"
    NUM_EXACT = "Exact"
    NUM_GREATER = "Complete"
    NUM_APPROX = "Approx"
    NUM_LOWER = "Detect"
```

Notice that `NUM_EXACT` and `EXACT` share the exact same underlying value, and so do each of the other three pairs. The names `NUM_APPROX`, `NUM_GREATER`, and `NUM_LOWER` exist mainly so your code reads clearly when you are working with a numeric field, but what actually changes the comparison behavior is the field's `IndexType`, `INTEGER` or `DOUBLE`, not the specific name you import. When a mode is applied to a numeric field, the same four underlying strategies get reinterpreted for numbers:

| String field | Numeric field | Meaning for numbers |
|---|---|---|
| `EXACT` | `NUM_EXACT` | The numbers must be identical |
| `APPROX` | `NUM_APPROX` | Score based on how close the two numbers are |
| `COMPLETE` | `NUM_GREATER` | The indexed value must be greater than the query value |
| `DETECT` | `NUM_LOWER` | The indexed value must be lower than the query value |

Using the `NUM_` names in your own code is good practice even though they resolve to the same values, it makes the intent obvious to anyone reading it later.

## 2. Setting up

A small product catalog with a price and a release year, loaded from `datasets/product_catalog_with_year.csv`, both meaningful numeric fields for different reasons: price is something you'd want proximity or threshold matching on, release year is something you might want to filter by "before" or "after."

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.config import TableConfig, TableFieldConfig, IndexType

df = pd.read_csv("datasets/product_catalog_with_year.csv")

schema = TableConfig(fields=[
    TableFieldConfig(column="product_name", index_type=IndexType.PHRASE),
    TableFieldConfig(column="unit_price", index_type=IndexType.DOUBLE),
    TableFieldConfig(column="release_year", index_type=IndexType.INTEGER)
])

index = TableIndexer.create_index(
    df=df,
    config_overrides=schema,
    tmp_dir="tmp_index"
)

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


,product_name,unit_price,release_year
0,Extended Battery Pack,24.99,2022
1,Portable Power Bank,39.50,2023
2,Motion Sensor Camera,59.00,2021
3,Smart Relay Switch,18.75,2024
4,Solar Panel Charger,89.99,2023


## 3. `NUM_APPROX`: how close is close enough

`NUM_APPROX` scores numeric closeness on a continuous scale, the nearer the indexed value is to your query value, the higher the score. This fits searches like "find products priced around $25," where you want the exact match to score best, but nearby prices to still show up with a meaningfully lower score rather than being excluded entirely.

In [3]:
from mbox.recall import TableRecallMode

price_proximity = index.match(
    unit_price=25.00,
    modes={"unit_price": TableRecallMode.NUM_APPROX},
    min_total_match_value=0,
    include_field_scores=True,
    max_results=5
)

price_proximity

,query_row,index_row,unit_price_candidate,product_name_candidate,release_year_candidate,overall_score,unit_price_score
0,0,0,24.99,Extended Battery Pack,2022,98,98


`"Extended Battery Pack"` at $24.99 should score close to perfect, it is almost exactly the query value. Products further away in price, like the $89.99 solar charger, should score progressively lower. This is the numeric equivalent of `APPROX` for text: not a strict yes or no, but a graded sense of "how close."

## 4. `NUM_EXACT`: the number must match exactly

Sometimes proximity is not what you want at all. If you are looking up a product by an exact release year for a compliance report, or matching a specific price point that has legal meaning, like a contractually fixed rate, you want strict equality, not "close enough."

In [4]:
exact_year_match = index.match(
    release_year=2023,
    modes={"release_year": TableRecallMode.NUM_EXACT},
    min_total_match_value=0,
    include_field_scores=True,
    max_results=5
)

exact_year_match

,query_row,index_row,release_year_candidate,product_name_candidate,unit_price_candidate,overall_score,release_year_score
0,0,1,2023,Portable Power Bank,39.50,100,100
1,0,4,2023,Solar Panel Charger,89.99,100,100


Only rows where `release_year` is precisely `2023` should score well here, `"Portable Power Bank"` and `"Solar Panel Charger"`. A product released in `2022` or `2024` is not "close" under `NUM_EXACT`, it is simply wrong, the same all-or-nothing behavior you saw with `EXACT` on text fields.

## 5. `NUM_GREATER`: at least this much

`NUM_GREATER` matches when the *indexed* value is greater than your query value. This is the natural fit for a minimum threshold, "show me products released after 2022," or "products priced above $30."

In [5]:
newer_than_2022 = index.match(
    release_year=2022,
    modes={"release_year": TableRecallMode.NUM_GREATER},
    min_total_match_value=0,
    include_field_scores=True,
    max_results=5
)

newer_than_2022

,query_row,index_row,release_year_candidate,product_name_candidate,unit_price_candidate,overall_score,release_year_score
0,0,0,2022,Extended Battery Pack,24.99,100,100
1,0,1,2023,Portable Power Bank,39.50,100,100
2,0,3,2024,Smart Relay Switch,18.75,100,100
3,0,4,2023,Solar Panel Charger,89.99,100,100
4,0,2,2021,Motion Sensor Camera,59.00,99,99


Products released in `2023` and `2024` should score well here, since their `release_year` is greater than the `2022` we queried. The `2021` and `2022` products should not, they fail the threshold entirely rather than scoring partway.

## 6. `NUM_LOWER`: no more than this much

`NUM_LOWER` is the mirror image: it matches when the indexed value is lower than your query value. This fits a maximum threshold, "show me products priced under $30."

In [6]:
under_30 = index.match(
    unit_price=30.00,
    modes={"unit_price": TableRecallMode.NUM_LOWER},
    min_total_match_value=0,
    include_field_scores=True,
    max_results=5
)

under_30

,query_row,index_row,unit_price_candidate,product_name_candidate,release_year_candidate,overall_score,unit_price_score
0,0,0,24.99,Extended Battery Pack,2022,100,100
1,0,3,18.75,Smart Relay Switch,2024,100,100


Only `"Extended Battery Pack"` at $24.99 and `"Smart Relay Switch"` at $18.75 should score well, both genuinely priced below $30. Everything priced at $30 or above should not, regardless of how close it is to the threshold, `NUM_LOWER` is a boundary, not a proximity measure.

## 7. Combining a numeric field with a text field

Recall from the previous notebook that a multi-field query needs every field you search to have at least some relevant signal, otherwise a row may not enter the candidate set at all. The same rule applies when mixing numeric and text fields. Let's search for a battery-related product under $30.

In [7]:
from mbox.recall import TableRecallConfig, TableRecallFieldConfig

combined_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(
            input_column="product_name",
            indexed_column="product_name",
            minimum_quality=0,
            weight=60,
            mode=TableRecallMode.APPROX
        ),
        TableRecallFieldConfig(
            input_column="unit_price",
            indexed_column="unit_price",
            minimum_quality=0,
            weight=40,
            mode=TableRecallMode.NUM_LOWER
        )
    ],
    max_results=5,
    min_total_match_value=0,
    include_field_scores=True
)

combined_results = index.match(
    queries=pd.DataFrame({"product_name": ["Battery Pack"], "unit_price": [30.00]}),
    config=combined_config
)

combined_results

,query_row,index_row,product_name_candidate,unit_price_candidate,overall_score,product_name_score,unit_price_score
0,0,0,Extended Battery Pack,24.99,95,92,100


`"Extended Battery Pack"` should come out on top here, it has strong text overlap with `"Battery Pack"` and its price genuinely falls under the $30 threshold. A product with a great name match but a price at or above $30 would fail the `NUM_LOWER` comparison on that field, pulling down its contribution to `overall_score` even if its name match were excellent.

## 8. Practical use cases

| Situation | Mode |
|---|---|
| "Find records with a value close to X" | `NUM_APPROX` |
| "This field must equal exactly X" | `NUM_EXACT` |
| "At least X" (minimum price, minimum age, earliest date) | `NUM_GREATER` |
| "No more than X" (maximum price, latest date, budget ceiling) | `NUM_LOWER` |
| "Between X and Y" | Two `TableRecallFieldConfig` entries on the same underlying value, one `NUM_GREATER` for the lower bound and one `NUM_LOWER` for the upper bound |

That last row is worth calling out: M|BOX does not have a single "range" mode, but you can express a range by combining a lower-bound check and an upper-bound check as two separate field configs, each contributing to the same overall score.

## Next steps

- **`04-reusable_recall_configs_as_json.ipynb`** - save a full recall configuration, numeric modes, weights, and thresholds included, and reuse it across a pipeline

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*